# Phase 4 : Évaluation et Validation
## Projet de Détection d'Intrusions Réseau — Option A
### Dataset : UNSW-NB15

**Objectifs :**
1. Courbes ROC et AUC (one-vs-rest, 10 classes)
2. Courbes Precision-Recall
3. Analyse FP/FN avec impact métier
4. Tests de robustesse
5. Synthèse finale

---
## 0. Initialisation — Chargement des artefacts Phase 3

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Bibliotheques chargees.')

Bibliotheques chargees.


In [2]:
# Datasets
X_train = pd.read_csv('data/processed/X_train.csv')
y_train = pd.read_csv('data/processed/y_train.csv').squeeze()
X_val   = pd.read_csv('data/processed/X_val.csv')
y_val   = pd.read_csv('data/processed/y_val.csv').squeeze()
X_test  = pd.read_csv('data/processed/X_test.csv')
y_test  = pd.read_csv('data/processed/y_test.csv').squeeze()

print(f'X_train : {X_train.shape}')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')

X_train : (65865, 42)
X_val   : (16467, 42)
X_test  : (175341, 42)


In [3]:
# Pipeline de pretraitement
pipeline           = joblib.load('data/processed/preprocessing_pipeline.pkl')
scaler             = pipeline['scaler']
label_encoders     = pipeline['label_encoders']
le_target          = pipeline['le_target']
feature_cols_final = pipeline['feature_cols_final']
cap_limits         = pipeline['cap_limits']
class_names        = pipeline['target_classes']

print(f'Classes ({len(class_names)}) : {class_names}')
print(f'Features : {len(feature_cols_final)}')

Classes (10) : ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']
Features : 42


In [4]:
# Modeles entraines
model_files = {
    'Logistic Regression'      : 'models/logistic_regression.pkl',
    'Decision Tree'            : 'models/decision_tree.pkl',
    'Random Forest'            : 'models/random_forest.pkl',
    'XGBoost'                  : 'models/xgboost.pkl',
    'MLP Neural Network'       : 'models/mlp_neural_network.pkl',
    'Random Forest Optimise'   : 'models/random_forest_optimisé.pkl',
    'XGBoost Optimise'         : 'models/xgboost_optimisé.pkl',
}

models = {}
for name, path in model_files.items():
    models[name] = joblib.load(path)
    print(f'  [OK] {name}')

# Meilleur modele — XGBoost Optimise (F1=0.7567 sur test)
best_model_name = 'XGBoost Optimise'
best_model      = models[best_model_name]
print(f'Meilleur modele : {best_model_name}')

  [OK] Logistic Regression


  [OK] Decision Tree


  [OK] Random Forest


  [OK] XGBoost
  [OK] MLP Neural Network


  [OK] Random Forest Optimise


  [OK] XGBoost Optimise
Meilleur modele : XGBoost Optimise


In [5]:
# Benchmark Phase 3
benchmark = pd.read_csv('data/processed/benchmark_results.csv')
print(benchmark.to_string(index=False))

                  Modèle  Accuracy (Val)  F1 (Val)  Accuracy (Test)  Precision (Test)  Recall (Test)  F1 (Test)  Temps (s)
     Logistic Regression          0.6798    0.7244           0.7012            0.7740         0.7012     0.7193      35.12
           Decision Tree          0.8321    0.8515           0.7348            0.7677         0.7348     0.7277       0.74
           Random Forest          0.8671    0.8789           0.7628            0.8012         0.7628     0.7526       4.00
                 XGBoost          0.8643    0.8794           0.7617            0.7837         0.7617     0.7502      29.70
      MLP Neural Network          0.8003    0.8245           0.7338            0.7905         0.7338     0.7307     110.00
Random Forest (Optimisé)          0.8669    0.8793           0.7632            0.8000         0.7632     0.7551       3.21
      XGBoost (Optimisé)          0.8582    0.8740           0.7636            0.7994         0.7636     0.7567       8.35


In [6]:
# Verification finale
print('=' * 50)
print('   INITIALISATION PHASE 4 — OK')
print('=' * 50)
print(f'  Train      : {X_train.shape[0]:>8,} connexions | {X_train.shape[1]} features')
print(f'  Validation : {X_val.shape[0]:>8,} connexions')
print(f'  Test       : {X_test.shape[0]:>8,} connexions')
print(f'  Modeles    : {len(models)}')
print(f'  Classes    : {len(class_names)}')
print('Pret pour la Phase 4.')

   INITIALISATION PHASE 4 — OK
  Train      :   65,865 connexions | 42 features
  Validation :   16,467 connexions
  Test       :  175,341 connexions
  Modeles    : 7
  Classes    : 10
Pret pour la Phase 4.
